In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# A2A Agent Card

An Agent Card is an agent's public description: its name, where to reach it, what it can do (skills, with example requests), how it talks (streaming, push notifications) and how a client authenticates. A client reads the card before sending any work, which is how one agent discovers another. A server publishes it at a well-known path, now `/.well-known/agent-card.json` (the book's `/.well-known/agent.json` is the older name).

The book's WeatherBot card (p. 210) was written against an early version of the protocol. A2A has since reached 1.0, and the card no longer parses: the next cell shows the SDK's error and then the same card in 1.0 field names.

| The book's card | A2A 1.0 |
|---|---|
| `"url"` | `"supportedInterfaces": [{"url", "protocolBinding", "protocolVersion"}]`: one agent can offer several transports |
| `"authentication": {"schemes": ["apiKey"]}` | `"securitySchemes"` (how: an API key in a named header) plus `"securityRequirements"` (which scheme a request must satisfy) |
| `"stateTransitionHistory"` capability | removed |

The card cell calls no model, so there is no Ollama version.

In [3]:
# Book listing: Chapter 15, Agent Card for a WeatherBot, p. 210. No model.
# The card is the book's JSON verbatim (true/false written as Python True/False). Parsing
# it with the A2A SDK checks it against the current protocol's schema.
from google.protobuf.json_format import MessageToJson, ParseDict, ParseError
from a2a.types import AgentCard

weatherbot_card = {
  "name": "WeatherBot",
  "description": "Provides accurate weather forecasts and historical data.",
  "url": "http://weather-service.example.com/a2a",
  "version": "1.0.0",
  "capabilities": {
    "streaming": True,
    "pushNotifications": False,
    "stateTransitionHistory": True
  },
  "authentication": {
    "schemes": [
      "apiKey"
    ]
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "get_current_weather",
      "name": "Get Current Weather",
      "description": "Retrieve real-time weather for any location.",
      "inputModes": [
        "text"
      ],
      "outputModes": [
        "text"
      ],
      "examples": [
        "What's the weather in Paris?",
        "Current conditions in Tokyo"
      ],
      "tags": [
        "weather",
        "current",
        "real-time"
      ]
    },
    {
      "id": "get_forecast",
      "name": "Get Forecast",
      "description": "Get 5-day weather predictions.",
      "inputModes": [
        "text"
      ],
      "outputModes": [
        "text"
      ],
      "examples": [
        "5-day forecast for New York",
        "Will it rain in London this weekend?"
      ],
      "tags": [
        "weather",
        "forecast",
        "prediction"
      ]
    }
  ]
}

try:
    ParseDict(weatherbot_card, AgentCard())
except ParseError as error:
    print(f"The book's card is not a valid A2A 1.0 card:\n  {error}\n")

# The same card in A2A 1.0: the endpoint moves into supported_interfaces, the API key
# into security_schemes + security_requirements, and stateTransitionHistory is gone.
current = {k: v for k, v in weatherbot_card.items() if k not in ("url", "authentication")}
current["capabilities"] = {"streaming": True, "pushNotifications": False}
current["supportedInterfaces"] = [
    {"url": "http://weather-service.example.com/a2a", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}
]
current["securitySchemes"] = {
    "apiKey": {"apiKeySecurityScheme": {"location": "header", "name": "X-API-Key"}}
}
current["securityRequirements"] = [{"schemes": {"apiKey": {}}}]

card = ParseDict(current, AgentCard())
print(f"Parsed: {card.name}, skills {[s.id for s in card.skills]}, "
      f"endpoint {card.supported_interfaces[0].url}")

The book's card is not a valid A2A 1.0 card:
  Message type "lf.a2a.v1.AgentCard" has no field named "url" at "AgentCard".
 Available Fields(except extensions): "['name', 'description', 'supportedInterfaces', 'provider', 'version', 'documentationUrl', 'capabilities', 'securitySchemes', 'securityRequirements', 'defaultInputModes', 'defaultOutputModes', 'skills', 'signatures', 'iconUrl']"

Parsed: WeatherBot, skills ['get_current_weather', 'get_forecast'], endpoint http://weather-service.example.com/a2a


# Google ADK A2A server

The book's code example (pp. 217-219) is two excerpts from the `birthday_planner_adk` sample in `a2a-samples`: `create_agent` builds an ADK calendar agent, and `main()` wraps it in an A2A server: Agent Card, `Runner`, an executor that translates between A2A tasks and ADK events, a request handler, a Starlette app, `uvicorn.run`. The excerpts are not runnable as printed (no imports, `main()` called without its arguments), and the book never shows a client. The book's two JSON-RPC requests (pp. 213-214) are that missing client.

Ported:

| The book | Here |
|---|---|
| `CalendarToolset(client_id=..., client_secret=...)` | a mock `check_availability` tool with one busy slot tomorrow 10:00-11:00. The real toolset needs a Google OAuth client and a calendar, and the `/authenticate` route only exists to finish that OAuth flow, so it is dropped too |
| `model='gemini-2.0-flash-001'` | retired; `gemini-flash-latest` |
| `ADKAgentExecutor(runner, agent_card)` | a class from the sample, not a library. ADK now ships one: `A2aAgentExecutor(runner=runner)` |
| `A2AStarletteApplication(agent_card=..., http_handler=...)` | removed in `a2a-sdk` 1.x; `create_agent_card_routes` + `create_jsonrpc_routes` give the same routes |
| `AgentCard(url=..., defaultInputModes=...)` | `supported_interfaces=[AgentInterface(url=...)]`, `default_input_modes=...`: the 1.0 card is a protobuf message with snake_case fields |
| `uvicorn.run(app, ...)`, `asyncio.run(create_agent(...))` | both block or start an event loop, which a notebook already has. The server runs as a background task in the kernel, the cell talks to it, then stops it |
| `"method": "sendTask"` / `"sendTaskSubscribe"` | `SendMessage` / `SendStreamingMessage` (the protocol went through `message/send` / `message/stream` on the way). The book's two requests are sent as printed, to show the rejection |
| `"params": {"id": "task-001", "sessionId": ...}` | the client no longer picks the task id; the server assigns `id` and `contextId`. The client names its message (`messageId`), and `acceptedOutputModes` / `historyLength` move into `configuration` |

Every request also carries an `A2A-Version: 1.0` header. Without it the server treats the request as protocol 0.3 and answers a 1.0 method with `Version mismatch`.

The two `Request Error ... Code=-32601` lines in the cell's stderr are the server logging the book's two requests it rejected; they are expected. ADK can also build the whole server from an agent in one call, `to_a2a(agent)`; the cells spell it out to match the book's `main()`.

The next cell holds the tool, the card, the server and the client, with no model. The Gemini cell and the Ollama cell each build the agent and serve it on their own port, so either can run on its own.

In [4]:
# Book listings: Chapter 15, Hands-On Code Example, pp. 217-219 - the calendar agent
# (adk_agent.py) and the server that exposes it over A2A (__main__.py), from the
# a2a-samples birthday_planner_adk sample (Apache-2.0).
# This cell holds the tool, the Agent Card, the server and a client, and no model. The
# Gemini cell and the Ollama cell below each build the agent and call run_a2a_demo.
import asyncio
import datetime
import json
import os

import httpx
import uvicorn
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill
from google.adk.a2a.executor.a2a_agent_executor import A2aAgentExecutor  # book: ADKAgentExecutor
from google.adk.agents import LlmAgent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from sse_starlette.sse import AppStatus
from starlette.applications import Starlette

os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")
# ADK marks its A2A bridge experimental and warns on every first use of each of ~10
# functions. This is ADK's own switch for those warnings, read at call time.
os.environ.setdefault("ADK_SUPPRESS_A2A_EXPERIMENTAL_FEATURE_WARNINGS", "true")

HOST = "127.0.0.1"

# The streaming responses come from sse-starlette, which watches the first uvicorn server
# it sees and, once that server stops, flags the whole process as shutting down. In a
# kernel that outlives the server, every later stream (a second server, or this cell run
# twice) then closes after its first event. Switching that off is safe here: the client
# reads each stream to its end before the cell stops the server.
AppStatus.disable_automatic_graceful_drain()


# --- The agent (adk_agent.py) ---
# The book's CalendarToolset needs a Google OAuth client and a real calendar. This mock
# has one busy slot, tomorrow 10:00-11:00, and answers the skill's example question.
TOMORROW = datetime.date.today() + datetime.timedelta(days=1)
BUSY = [(datetime.datetime.combine(TOMORROW, datetime.time(10)),
         datetime.datetime.combine(TOMORROW, datetime.time(11)), "Team sync")]


def check_availability(start: str, end: str) -> dict:
    """Checks the user's primary calendar for events between two times.

    Args:
        start: Start of the period, an RFC3339 timestamp.
        end: End of the period, an RFC3339 timestamp.
    """
    print(f"  [tool] check_availability({start!r}, {end!r})")
    begin = datetime.datetime.fromisoformat(start).replace(tzinfo=None)
    finish = datetime.datetime.fromisoformat(end).replace(tzinfo=None)
    clashes = [title for s, e, title in BUSY if s < finish and begin < e]
    return {"free": not clashes, "conflicting_events": clashes}


def create_agent(model, **options) -> LlmAgent:
    """Constructs the ADK agent.

    The book's version is async and takes client_id/client_secret, because it awaits
    CalendarToolset(...).get_tools(); with the mock tool it takes the model instead.
    """
    return LlmAgent(
        model=model,
        name='calendar_agent',
        description="An agent that can help manage a user's calendar",
        instruction=f"""
You are an agent that can help manage a user's calendar.

Users will request information about the state of their calendar or to make changes to
their calendar. Use the provided tools for interacting with the calendar API.

If not specified, assume the calendar the user wants is the 'primary' calendar.

When using the Calendar API tools, use well-formed RFC3339 timestamps.

Today is {datetime.datetime.now()}.
""",
        tools=[check_availability],  # book: await toolset.get_tools()
        **options,
    )


# --- The server (__main__.py) ---
def build_agent_card(port: int) -> AgentCard:
    """The book's skill and card, in A2A 1.0 field names."""
    skill = AgentSkill(
        id='check_availability',
        name='Check Availability',
        description="Checks a user's availability for a time using their Google Calendar",
        tags=['calendar'],
        examples=['Am I free from 10am to 11am tomorrow?'],
    )
    return AgentCard(
        name='Calendar Agent',
        description="An agent that can manage a user's calendar",
        # book: url=f'http://{host}:{port}/'
        supported_interfaces=[AgentInterface(
            url=f'http://{HOST}:{port}/', protocol_binding='JSONRPC', protocol_version='1.0')],
        version='1.0.0',
        default_input_modes=['text'],
        default_output_modes=['text'],
        capabilities=AgentCapabilities(streaming=True),
        skills=[skill],
    )


def build_a2a_app(adk_agent: LlmAgent, port: int) -> Starlette:
    """The book's main() without the OAuth route and uvicorn.run."""
    agent_card = build_agent_card(port)
    runner = Runner(
        app_name=adk_agent.name,  # book: agent_card.name; ADK wants an identifier
        agent=adk_agent,
        artifact_service=InMemoryArtifactService(),
        session_service=InMemorySessionService(),
        memory_service=InMemoryMemoryService(),
    )
    agent_executor = A2aAgentExecutor(runner=runner)
    request_handler = DefaultRequestHandler(
        agent_executor=agent_executor, task_store=InMemoryTaskStore(), agent_card=agent_card
    )
    # book: A2AStarletteApplication(agent_card=..., http_handler=...).routes()
    routes = create_agent_card_routes(agent_card) + create_jsonrpc_routes(request_handler, '/')
    return Starlette(routes=routes)


# --- The client: the book's two JSON-RPC requests, then the same in A2A 1.0 ---
BOOK_SYNC_REQUEST = {
  "jsonrpc": "2.0",
  "id": "1",
  "method": "sendTask",
  "params": {
    "id": "task-001",
    "sessionId": "session-001",
    "message": {
      "role": "user",
      "parts": [
        {
          "type": "text",
          "text": "What is the exchange rate from USD to EUR?"
        }
      ]
    },
    "acceptedOutputModes": ["text/plain"],
    "historyLength": 5
  }
}

BOOK_STREAMING_REQUEST = {
  "jsonrpc": "2.0",
  "id": "2",
  "method": "sendTaskSubscribe",
  "params": {
    "id": "task-002",
    "sessionId": "session-001",
    "message": {
      "role": "user",
      "parts": [
        {
          "type": "text",
          "text": "What's the exchange rate for JPY to GBP today?"
        }
      ]
    },
    "acceptedOutputModes": ["text/plain"],
    "historyLength": 5
  }
}


def current_request(method: str, request_id: str, text: str) -> dict:
    """A2A 1.0 form: the server assigns task and context ids, the client names its message."""
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": {
            "message": {"messageId": f"msg-{request_id}", "role": "ROLE_USER", "parts": [{"text": text}]},
            "configuration": {"acceptedOutputModes": ["text/plain"], "historyLength": 5},
        },
    }


def print_result(result: dict) -> None:
    """Prints the task state and the agent's reply from a SendMessage result."""
    task = result["task"]
    print(f"  task {task['id'][:8]}... state {task['status']['state']}")
    for artifact in task.get("artifacts", []):
        for part in artifact["parts"]:
            print(f"  reply: {part['text'].strip()}")


async def run_a2a_demo(adk_agent: LlmAgent, port: int) -> None:
    """Serves the agent over A2A, then talks to it as a remote client would."""
    server = uvicorn.Server(uvicorn.Config(build_a2a_app(adk_agent, port), host=HOST, port=port, log_level="warning"))
    serving = asyncio.create_task(server.serve())
    while not server.started:
        await asyncio.sleep(0.1)
    base = f"http://{HOST}:{port}"
    try:
        # Without this header the server treats a request as A2A 0.3 and rejects 1.0 methods.
        async with httpx.AsyncClient(timeout=120, headers={"A2A-Version": "1.0"}) as client:
            card = (await client.get(f"{base}/.well-known/agent-card.json")).json()
            print(f"Discovered '{card['name']}' with skills {[s['id'] for s in card['skills']]}")

            for book_request in (BOOK_SYNC_REQUEST, BOOK_STREAMING_REQUEST):
                print(f"\n>>> The book's request, method {book_request['method']}")
                reply = (await client.post(base, json=book_request)).json()
                print(f"  error: {reply['error']}")

            print("\n>>> SendMessage: Am I free from 10am to 11am tomorrow?")
            request = current_request("SendMessage", "3", "Am I free from 10am to 11am tomorrow?")
            print_result((await client.post(base, json=request)).json()["result"])

            print("\n>>> SendStreamingMessage: Am I free from 2pm to 3pm tomorrow?")
            request = current_request("SendStreamingMessage", "4", "Am I free from 2pm to 3pm tomorrow?")
            async with client.stream("POST", base, json=request) as response:
                async for line in response.aiter_lines():
                    if line.startswith("data:"):
                        event = json.loads(line[5:])["result"]
                        kind = next(iter(event))
                        detail = event[kind].get("status", {}).get("state", "")
                        print(f"  [sse] {kind} {detail}")
                        if kind == "artifactUpdate":
                            print(f"  reply: {event[kind]['artifact']['parts'][0]['text'].strip()}")
    finally:
        server.should_exit = True
        await serving

In [5]:
# The book's calendar agent on Gemini, served over A2A on port 10001.
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

calendar_agent = create_agent(
    'gemini-flash-latest',  # the book's gemini-2.0-flash-001 is retired
    generate_content_config=GenerateContentConfig(
        automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
    ),
)

await run_a2a_demo(calendar_agent, port=10001)

Request Error (ID: 1): Code=-32601, Message='Method not found'


Request Error (ID: 2): Code=-32601, Message='Method not found'


Discovered 'Calendar Agent' with skills ['check_availability']

>>> The book's request, method sendTask
  error: {'code': -32601, 'message': 'Method not found'}

>>> The book's request, method sendTaskSubscribe
  error: {'code': -32601, 'message': 'Method not found'}

>>> SendMessage: Am I free from 10am to 11am tomorrow?


  [tool] check_availability('2026-11-02T10:00:00Z', '2026-11-02T11:00:00Z')


  task 2eb216c2... state TASK_STATE_COMPLETED
  reply: No, you are not free tomorrow from 10:00 AM to 11:00 AM. You have a conflicting event scheduled: **Team sync**.

>>> SendStreamingMessage: Am I free from 2pm to 3pm tomorrow?
  [sse] task TASK_STATE_SUBMITTED
  [sse] statusUpdate TASK_STATE_WORKING


  [tool] check_availability('2026-11-02T14:00:00Z', '2026-11-02T15:00:00Z')
  [sse] statusUpdate TASK_STATE_WORKING
  [sse] statusUpdate TASK_STATE_WORKING


  [sse] statusUpdate TASK_STATE_WORKING
  [sse] artifactUpdate 
  reply: Yes, you are free from 2:00 PM to 3:00 PM tomorrow (November 2, 2026). There are no conflicting events scheduled.
  [sse] statusUpdate TASK_STATE_COMPLETED


## Google ADK + Ollama

In [6]:
# Same calendar agent, running on a local model served by Ollama through LiteLLM,
# behind the same A2A server on port 10002.
# Prerequisites:
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Needs only the shared cell above, not the Gemini cell, and no Google key.
# The client cannot tell which model answers: the Agent Card and the protocol are the same.
from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

ollama_calendar_agent = create_agent(LiteLlm(model="ollama_chat/qwen2.5:7b"))

await run_a2a_demo(ollama_calendar_agent, port=10002)

Request Error (ID: 1): Code=-32601, Message='Method not found'


Request Error (ID: 2): Code=-32601, Message='Method not found'


Discovered 'Calendar Agent' with skills ['check_availability']

>>> The book's request, method sendTask
  error: {'code': -32601, 'message': 'Method not found'}

>>> The book's request, method sendTaskSubscribe
  error: {'code': -32601, 'message': 'Method not found'}

>>> SendMessage: Am I free from 10am to 11am tomorrow?


  [tool] check_availability('2026-11-02T10:00:00+00:00', '2026-11-02T11:00:00+00:00')


  task c733ee61... state TASK_STATE_COMPLETED
  reply: You are not free between 10am and 11am tomorrow. There is a conflicting event titled "Team sync" during that time slot.

>>> SendStreamingMessage: Am I free from 2pm to 3pm tomorrow?
  [sse] task TASK_STATE_SUBMITTED
  [sse] statusUpdate TASK_STATE_WORKING


  [tool] check_availability('2026-11-02T14:00:00+00:00', '2026-11-02T15:00:00+00:00')
  [sse] statusUpdate TASK_STATE_WORKING
  [sse] statusUpdate TASK_STATE_WORKING


  [sse] statusUpdate TASK_STATE_WORKING
  [sse] artifactUpdate 
  reply: You are free from 2pm to 3pm tomorrow. There are no conflicting events in that time slot.
  [sse] statusUpdate TASK_STATE_COMPLETED
